In [1]:
import pandas as pd
from pandas.io.sas.sas_constants import dataset_offset

In [2]:
post = pd.read_csv('../data/raw/reddit/the-reddit-ethereum-dataset-posts.csv')
comments = pd.read_csv('../data/raw/reddit/the-reddit-ethereum-dataset-comments.csv')

In [9]:
# post data
print("post shape: ", post.shape)
print("\npost columns and data types", post.dtypes)

# comments dataset
print("comments shape: ", comments.shape)
print("\ncomments columns and data types", comments.dtypes)


post shape:  (479260, 12)

post columns and data types type              object
id                object
subreddit.id      object
subreddit.name    object
subreddit.nsfw      bool
created_utc        int64
permalink         object
domain            object
url               object
selftext          object
title             object
score              int64
dtype: object
comments shape:  (1132331, 10)

comments columns and data types type               object
id                 object
subreddit.id       object
subreddit.name     object
subreddit.nsfw       bool
created_utc         int64
permalink          object
body               object
sentiment         float64
score               int64
dtype: object


In [ ]:
# ==============================================
# Reddit Social Data Processing for FuseChain
# ==============================================
# This script:
# - Loads posts.csv and comments.csv
# - Filters for "Ethereum" + fraud-related keywords
# - Aggregates daily counts and average sentiment
# - Generates lag features (1, 3, 7 days)
# - Outputs a merged daily feature dataset
# ==============================================

import pandas as pd
import re

# === Step 1: Load datasets ===
posts_df = pd.read_csv("posts.csv")        # contains posts (title + selftext)
comments_df = pd.read_csv("comments.csv")  # contains comments (body text)

print("Posts dataset shape:", posts_df.shape)
print("Comments dataset shape:", comments_df.shape)

# === Step 2: Combine relevant text fields ===
posts_df['text'] = (posts_df['title'].fillna('') + ' ' + posts_df['selftext'].fillna('')).str.lower()
comments_df['text'] = comments_df['body'].fillna('').astype(str).str.lower()

# === Step 3: Define fraud-related keyword pattern ===
fraud_keywords = [
    'scam', 'phishing', 'fraud', 'hack', 'stolen',
    'rug pull', 'exploit', 'malicious', 'ponzi',
    'attack', 'breach', 'compromised wallet', 'rug'
]
pattern = re.compile('|'.join(fraud_keywords))

# === Step 4: Filter for Ethereum + fraud keywords ===
posts_filtered = posts_df[
    posts_df['text'].str.contains('ethereum', na=False) &
    posts_df['text'].str.contains(pattern, na=False)
]
comments_filtered = comments_df[
    comments_df['text'].str.contains('ethereum', na=False) &
    comments_df['text'].str.contains(pattern, na=False)
]

print(f"Filtered posts: {len(posts_filtered)} / {len(posts_df)}")
print(f"Filtered comments: {len(comments_filtered)} / {len(comments_df)}")

# === Step 5: Convert timestamps to daily UTC date ===
for df in [posts_filtered, comments_filtered]:
    df['date'] = pd.to_datetime(df['created_utc'], unit='s', utc=True).dt.date

# === Step 6: Aggregate daily counts ===
daily_posts = (
    posts_filtered.groupby('date')
    .size()
    .rename('reddit_post_mentions')
    .reset_index()
)
daily_comments = (
    comments_filtered.groupby('date')
    .size()
    .rename('reddit_comment_mentions')
    .reset_index()
)

# === Step 7: Merge post & comment daily counts ===
reddit_daily = pd.merge(daily_posts, daily_comments, on='date', how='outer').fillna(0)
reddit_daily['reddit_total_mentions'] = (
    reddit_daily['reddit_post_mentions'] + reddit_daily['reddit_comment_mentions']
)

# === Step 8: Aggregate average sentiment (if available) ===
if 'sentiment' in posts_filtered.columns:
    daily_sentiment = (
        posts_filtered.groupby('date')['sentiment']
        .mean()
        .rename('reddit_avg_sentiment')
        .reset_index()
    )
    reddit_daily = reddit_daily.merge(daily_sentiment, on='date', how='left')

# === Step 9: Sort by date & reset index ===
reddit_daily = reddit_daily.sort_values('date').reset_index(drop=True)

# === Step 10: Generate lag features (1-day, 3-day, 7-day) ===
for col in ['reddit_total_mentions', 'reddit_avg_sentiment']:
    if col in reddit_daily.columns:
        for lag in [1, 3, 7]:
            reddit_daily[f'{col}_lag{lag}'] = reddit_daily[col].shift(lag)

# === Step 11: Fill missing lag values with 0 ===
reddit_daily = reddit_daily.fillna(0)

# === Step 12: Preview and export ===
print("\n=== Reddit Daily Feature Summary ===")
print(reddit_daily.head(10))

# Save to CSV for later merging with market & on-chain data
reddit_daily.to_csv("reddit_daily_features.csv", index=False)
print("\n✅ Saved cleaned Reddit features to 'reddit_daily_features.csv'")
